In [16]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SparkApp").getOrCreate()

customers = spark.read.option("header", True).csv("customers.csv")
order_items = spark.read.option("header", True).csv("order_items.csv")
orders = spark.read.option("header", True).csv("orders.csv")
products = spark.read.option("header", True).csv("products.csv")
returns = spark.read.option("header", True).csv("returns.csv")

print("Customer Count:", customers.count())
print("Order Items Count:", order_items.count())
print("Orders Count:", orders.count())
print("Product Count:", products.count())
print("Returns Count:", returns.count())

Customer Count: 100000


Order Items Count: 3000000
Orders Count: 1000000
Product Count: 50000
Returns Count: 100000


In [29]:
result.coalesce(1) \
      .write \
      .mode("overwrite") \
      .option("header", True) \
      .csv("question 1")

In [26]:
customers.createOrReplaceTempView("customers")
order_items.createOrReplaceTempView("order_items")
orders.createOrReplaceTempView("orders")
products.createOrReplaceTempView("products")
returns.createOrReplaceTempView("returns")

result = spark.sql("""
SELECT
    p.category,
    ROUND(SUM(oi.quantity * oi.selling_price), 2) AS total_sales_amount
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_sales_amount DESC
""")

result.show()

[Stage 81:==============>                                           (1 + 3) / 4]

+--------------+------------------+
|      category|total_sales_amount|
+--------------+------------------+
|        Beauty|     7.626693059E8|
|Home & Kitchen|    7.5813887328E8|
|         Books|    7.4649077835E8|
|          Toys|     7.446190723E8|
|   Electronics|    7.4426650411E8|
|        Sports|    7.4333886813E8|
|      Clothing|    7.4192279457E8|
+--------------+------------------+



In [30]:
result.coalesce(1) \
      .write \
      .mode("overwrite") \
      .option("header", True) \
      .csv("question 2")

In [27]:
customers.createOrReplaceTempView("customers")
order_items.createOrReplaceTempView("order_items")
orders.createOrReplaceTempView("orders")
products.createOrReplaceTempView("products")

result = spark.sql("""
SELECT
    c.customer_id,
    c.customer_name,
    ROUND(SUM(oi.quantity * oi.selling_price), 2) AS total_purchase_amount
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_purchase_amount DESC
LIMIT 10
""")

result.show(truncate=False)

[Stage 89:==============>                                           (1 + 3) / 4]

+-----------+--------------+---------------------+
|customer_id|customer_name |total_purchase_amount|
+-----------+--------------+---------------------+
|93094      |Customer_93094|181569.68            |
|64560      |Customer_64560|169060.4             |
|23289      |Customer_23289|161573.8             |
|52275      |Customer_52275|153364.79            |
|61218      |Customer_61218|153067.55            |
|52034      |Customer_52034|152680.05            |
|40442      |Customer_40442|151037.32            |
|60528      |Customer_60528|148691.95            |
|84830      |Customer_84830|148363.84            |
|82593      |Customer_82593|148281.04            |
+-----------+--------------+---------------------+



In [31]:
result.coalesce(1) \
      .write \
      .mode("overwrite") \
      .option("header", True) \
      .csv("question 3")

In [32]:
result = spark.sql("""
WITH latest_year AS (
    SELECT MAX(YEAR(order_date)) AS yr
    FROM orders
)

SELECT
    YEAR(o.order_date) AS year,
    MONTH(o.order_date) AS month,
    ROUND(SUM(oi.quantity * oi.selling_price), 2) AS total_sales
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE YEAR(o.order_date) = (SELECT yr FROM latest_year)
GROUP BY
    YEAR(o.order_date),
    MONTH(o.order_date)
ORDER BY month
""")

result.show()

[Stage 145:>                                                        (0 + 4) / 4]

+----+-----+--------------+
|year|month|   total_sales|
+----+-----+--------------+
|2024|    1|4.4457777576E8|
|2024|    2| 4.153661442E8|
|2024|    3|4.4362824541E8|
|2024|    4|4.2782097434E8|
|2024|    5|4.4481061895E8|
|2024|    6|4.3170515406E8|
|2024|    7|4.4367051912E8|
|2024|    8|4.4109517702E8|
|2024|    9|4.3107152608E8|
|2024|   10|4.4136378931E8|
|2024|   11|4.3362336404E8|
|2024|   12|4.4271290835E8|
+----+-----+--------------+

